In [ ]:
num_frames = int(numpy.ceil(float(numpy.abs(signal_length - frame_length)) / frame_step))  # Make sure that we have at least 1 frame
pad_signal_length = num_frames * frame_step + frame_length
z = numpy.zeros((pad_signal_length - signal_length))
pad_signal = numpy.append(emphasized_signal, z) # Pad Signal to make sure that all frames have equal number of samples without truncating any samples from the original signal


In [3]:
# !pip3 install tf-models-official==2.9.2
# ! pip uninstall python-levenshtein 0.20.9 --yes
# ! pip install jiwer

/bin/bash: /opt/conda/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /opt/conda/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [4]:
import tensorflow as tf
import math
# import tensorflow_models as tfm
from tensorflow.keras.layers import Conv1D, MaxPooling1D, TimeDistributed, Dense, GRU, Add, LayerNormalization, ReLU
import numpy as np
from jiwer import wer
print(tf.__version__)

2.9.2


In [5]:
class Linear(tf.keras.layers.Layer):
    def __init__(self, in_feature, out_feature, name=None):
        super().__init__()
        
        self.in_feature = in_feature
        self.out_feature = out_feature
        self.nameL = name if name is not None else self.__class__.__name__
        
        w_init = tf.random_normal_initializer()
        self.w = tf.Variable(initial_value=w_init(shape=(self.in_feature, self.out_feature),
                                                  dtype='float32'), trainable=True)
        
    def call(self, x):
        x = tf.matmul(x, self.w)
        return x


class RelativeMultiHeadAttenion(tf.keras.layers.Layer):
    def __init__(self, heads, d_model, max_len=500, **kwargs):
        super().__init__(**kwargs)
        assert d_model % heads ==0, "Model dim should be divisable by num of heads"
        
        self.max_len = max_len
        self.heads   = heads
        self.d_model = d_model
        self.d     = self.d_model // self.heads
        
        self.WQ = Linear(self.d_model, self.d_model)
        self.WK = Linear(self.d_model, self.d_model)
        self.WV = Linear(self.d_model, self.d_model)
        w_init = tf.random_normal_initializer()
        self.Er = tf.Variable(initial_value=w_init(shape=(self.max_len, self.d), dtype='float32'), trainable=True)
        self.WMerge = Linear(self.d_model, self.d_model)
        

    def call(self, query, key, value, mask=False, training=False):
        '''mask is 4d rank tensor of shape [N, time_step, time_step]'''
        BATCH_SIZE, q_len, _ = query.shape
        BATCH_SIZE, k_len, _ = key.shape
        BATCH_SIZE, v_len, _ = value.shape
        
        assert k_len == v_len and v_len == q_len, f"time step should be equal. but get query: {q_len} and key: {k_len} and value: {v_len}"
        
        query = self.WQ(query)     
        key = self.WK(key) 
        value = self.WV(value) 
        
        query = tf.reshape(query, [BATCH_SIZE, self.heads, -1, self.d])
        key_t = tf.reshape(key, [BATCH_SIZE, self.heads, self.d, -1])
        value = tf.reshape(value, [BATCH_SIZE, self.heads, -1, self.d])
        
        start = self.max_len - q_len
        Er_t = tf.transpose(self.Er[start:, :], perm=[1, 0] )
        
        QEr = tf.linalg.matmul(query, Er_t)
        # QEr.shape = (batch_size, num_heads, q_len, q_len)
        Srel = self.skew(QEr)
        # Srel.shape = (batch_size, num_heads, q_len, q_len)
        
        # query: BATCH_SIZE, heads, q_len, d
        # key_t: BATCH_SIZE, heads, d, k_len
        QK_t = tf.linalg.matmul(query, key_t)
        
        # QK_t: BATCH_SIZE, heads, q_len, q_len
        
        attn = (QK_t + Srel) / tf.math.sqrt(float(query.shape[-1]))
        if mask:
            mask = tf.linalg.band_part(tf.ones((q_len, q_len)), 0, -1) == 0
            mask = tf.expand_dim(mask, axis=0)
            mask = tf.expand_dim(mask, axis=0)
            # mask.shape = (1, 1, q_len, q_len)
            attn = mask * attn
            # attn.shape = (batch_size, num_heads, q_len, q_len)
        attn = tf.nn.softmax(attn, axis=-1)
        out = tf.linalg.matmul(attn, value)
        # out.shape = (batch_size, num_heads, q_len, d_head)
        out = tf.transpose(out, perm=[0, 2, 1, 3] )
        # out.shape == (batch_size, q_len, num_heads, d_head)
        out = tf.reshape(out, [BATCH_SIZE, -1, self.d_model])
        # out.shape == (batch_size, q_len, d_model)
        out = self.WMerge(out)
        if training:
            return tf.nn.dropout(out, rate =0.2)
        return out
    
    def skew(self, QEr):
        # QEr.shape = (batch_size, num_heads, seq_len, seq_len)
        
        padded = tf.pad(QEr, [[0, 0], [0, 0], [0, 0], [1, 0]])
        # padded.shape = (batch_size, num_heads, seq_len, 1 + seq_len)
        
        batch_size, num_heads, num_rows, num_cols = padded.shape
        reshaped = tf.reshape(padded, [batch_size, num_heads, num_cols, num_rows])
        # reshaped.size = (batch_size, num_heads, 1 + seq_len, seq_len)
        Srel = reshaped[:, :, 1:, :]
        # Srel.shape = (batch_size, num_heads, seq_len, seq_len)
        return Srel

In [6]:
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, heads, d_model, max_len=22000):
        super().__init__()
        self.heads       = heads
        self.d_model     = d_model
        self.max_len     = max_len
        self.RMHA        = RelativeMultiHeadAttenion(self.heads, self.d_model, self.max_len)
        self.risdual     = tf.keras.Sequential([Dense(2*self.d_model), ReLU(), Dense(self.d_model)])
        self.add         = Add()
        self.layer_norm1 = LayerNormalization()
        self.f1          = tf.keras.Sequential([Dense(self.d_model), ReLU()])
        self.layer_norm2 = LayerNormalization()
        
    def call(self, x, training=False):
        z = self.RMHA(x, x, x, training=training)
        x = self.risdual(x)
        x = self.add([x, z]); del z
        x = self.layer_norm1(x)
        
        z = self.f1(x)
        x = self.add([x, z]); del z
        x = self.layer_norm1(x)
        
        return x
        

class Encoder(tf.keras.layers.Layer):
    def __init__(self, heads, d_model, l=1, max_len=22000):
        super().__init__()
        self.heads   = heads
        self.d_model = d_model
        self.max_len = max_len
        self.L       = l
        
        self.enc1 = tf.keras.Sequential([EncoderLayer(heads, d_model, max_len) for _ in range(self.L)])
    
    def call(self, x, *args):
        x = self.enc1(x)
        return x

- windowing the signal 
    - window size = (20-40) ms 25 is standard
    - window step = 10 ms
- 
-

In [7]:
def calculate_output_length(length_in, kernel_size, stride=1, padding=0, dilation=1):
    return (length_in + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1
calculate_output_length(657, 2, 3)

219

In [8]:
class Wave2Context(tf.keras.Model):
    def __init__(self, sr, *args, **kwargs):
        super().__init__()
        self.sr = sr
                                                                                                                        #32000
        self.featuer_extractor = tf.keras.Sequential([Conv1D(filters=10, kernel_size=11, strides=3, activation='relu'), #10664
                                          Conv1D(filters=15, kernel_size=11, strides=1, activation='relu'),             #10654
                                          TimeDistributed(MaxPooling1D(2, 2)),                                          #5327
                                          Conv1D(filters=20, kernel_size=9, strides=2,  activation='relu'),             #2660
                                          Conv1D(filters=25, kernel_size=7, strides=2, activation='relu'),              #1327
                                          TimeDistributed(MaxPooling1D(2, 2)),                                          #663
                                          Conv1D(filters=30, kernel_size=5, strides=1, activation='relu'),              #659
                                          Conv1D(filters=40, kernel_size=3, strides=1, activation='relu'),              #657
                                          TimeDistributed(MaxPooling1D(2, 3))])                                         #219
                                         
        self.projection = Dense(256)                                                                                     #256
        
        self.pos_encoder = GRU(512, return_sequences=True)
        self.encoder     = Encoder(16, 512, 10)
        
    
    def call(self, x, training=False):
        # x.shape: B, step
        assert len(x.shape) <= 2, "the shape not match"
        assert x.shape[1] >= self.sr*2, f'Minimum length of wave is 2 second ({2 * self.sr})'
        
        Batch_size, step = x.shape
        x = x[:, :step - step % (self.sr*2)]
        x = tf.reshape(x, [Batch_size, step//(self.sr*2), -1, 1])
        # x.shape: B, n_second / 2, self.sr *2, 1
        print(x.shape)
        x = self.featuer_extractor(x)
        print(x.shape)
        # x.shape: B, n_second, 163, 16
        x = tf.reshape(x, [Batch_size, 40*(step//(self.sr*2)), -1])
        print(x.shape)
        # x.shape: B, n_second * 16, 163
        x = self.projection(x)
        # x.shape: B, n_second * 16, 100
        print(x.shape)
        
        x = self.pos_encoder(x, training=training)
        print(x.shape)
        x = self.encoder(x)
        print(x.shape)
        
        return x
    
def CTCLoss(y_true, y_pred):
    # Compute the training-time loss value
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")

    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")

    loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
    return loss

In [9]:
d = np.random.rand(1, 1*32000)
x = Wave2Context(16000)
x(d);
x.summary()

2023-02-21 23:46:19.186282: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/cuda/lib:/usr/local/lib/x86_64-linux-gnu:/usr/local/nvidia/lib:/usr/local/nvidia/lib64::/opt/conda/lib
2023-02-21 23:46:19.186333: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)


(1, 200, 32000, 1)
(1, 200, 219, 40)
(1, 8000, 219)
(1, 8000, 256)
(1, 8000, 512)


2023-02-21 23:46:28.972888: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 4096000000 exceeds 10% of free system memory.
2023-02-21 23:46:29.747535: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 4096512000 exceeds 10% of free system memory.
2023-02-21 23:46:30.276635: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 4096000000 exceeds 10% of free system memory.
2023-02-21 23:46:30.935570: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 4096000000 exceeds 10% of free system memory.
2023-02-21 23:46:31.677571: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 4096000000 exceeds 10% of free system memory.


(1, 8000, 512)
Model: "wave2_context"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential (Sequential)     (1, 200, 219, 40)         15450     
                                                                 
 dense (Dense)               multiple                  56320     
                                                                 
 gru (GRU)                   multiple                  1182720   
                                                                 
 encoder (Encoder)           multiple                  30663680  
                                                                 
Total params: 31,918,170
Trainable params: 31,918,170
Non-trainable params: 0
_________________________________________________________________


##

In [10]:
16*60

960

In [11]:
2*200/60

6.666666666666667

In [12]:
(220235455/16000)/60

229.41193229166666

In [13]:
1 * 63 *16000

1008000

In [14]:
# A callback class to output a few transcriptions during training
class CallbackEval(keras.callbacks.Callback):
    """Displays a batch of outputs after every epoch."""

    def __init__(self, dataset):
        super().__init__()
        self.dataset = dataset

    def on_epoch_end(self, epoch: int, logs=None):
        predictions = []
        targets = []
        for batch in self.dataset:
            X, y = batch
            batch_predictions = model.predict(X)
            batch_predictions = decode_batch_predictions(batch_predictions)
            predictions.extend(batch_predictions)
            for label in y:
                label = (
                    tf.strings.reduce_join(num_to_char(label)).numpy().decode("utf-8")
                )
                targets.append(label)
        wer_score = wer(targets, predictions)
        print("-" * 100)
        print(f"Word Error Rate: {wer_score:.4f}")
        print("-" * 100)
        for i in np.random.randint(0, len(predictions), 2):
            print(f"Target    : {targets[i]}")
            print(f"Prediction: {predictions[i]}")
            print("-" * 100)

NameError: name 'keras' is not defined